# 23 - Era 5x2 split: does similarity weighting help when hospitals come in twin pairs?

Follow-up to notebook 21 Part B. `era_cluster_split(n_clients=10, n_eras=5)`: the timeline is cut into **5 eras**, each era has
**2 hospitals** (random patients from that era). Hospitals in the same era are alike, hospitals in different eras differ
(offline diagnostic: 1.78x random label heterogeneity once patient history is fixed, vs 1.08x for org10; see `scripts/2026-11-22-realistic-splits.py`).

Same six arms as notebook 21 Part B, same budget (local-only 40 epochs, every arm ends with the same 8-epoch fine-tune):
FedAvg (+ local-only), v3 topology fingerprint, v3 cross-antibiogram fingerprint, shuffled control, **oracle** (equal weights
within the true era), SimFT. The oracle groups are `raw label // 2`.

**Seeds:** 42, 43, 44. **Runtime (rough, Colab T4):** ~1 to 1.5 h. Checkpointed after every seed; re-run cell 4 to resume.
Needs no code changes beyond what is already on `feat/budget-and-era`.

In [ ]:
# 1) Environment: Colab (clone + install + mount Drive) or local (.venv)
import os, sys
ON_COLAB = 'google.colab' in sys.modules
BRANCH = 'feat/budget-and-era'
if ON_COLAB:
    !git clone -b main https://github.com/RawEgg6/Capstone-amr-fed.git 2>/dev/null || (cd Capstone-amr-fed && git fetch && git checkout main && git pull)
    !cd /content/Capstone-amr-fed && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}
    !cd /content/Capstone-amr-fed && git log -1 --oneline
    !pip install -q torch_geometric 'flwr[simulation]' networkx scipy
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['ARMD_DIR'] = '/content/drive/MyDrive/ARMD'   # edit to your folder if different
else:
    os.environ.setdefault('ARMD_DIR', '/Users/vikramadityasharma/Library/CloudStorage/GoogleDrive-vikramaditya.raunak@gmail.com/.shortcut-targets-by-id/1uVx5PmldPyJBMtnB9Lkl9N1rNBQrmJt-/ARMD')
import torch, ray, flwr, torch_geometric
print('colab:', ON_COLAB, '| python', sys.version.split()[0], '| torch', torch.__version__, '| flwr', flwr.__version__,
      '| ray', ray.__version__, '| device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('ARMD_DIR:', os.environ['ARMD_DIR'])

In [ ]:
# 2) Sanity: data resolves + this branch has the notebook-21 modules
from pathlib import Path
if ON_COLAB:
    SRC = Path('/content/Capstone-amr-fed/src')
else:
    SRC = next(p / 'src' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'amr_fed').is_dir())
sys.path.insert(0, str(SRC))
from amr_fed import config
D = Path(config.DATA_DIR)
print('DATA_DIR:', D, '| exists:', D.exists())
assert D.exists(), 'ARMD_DIR is wrong - fix cell 1 and re-run.'
from amr_fed.federated import budget_runs as BR, budget as B, final_protocol as P
from amr_fed.partition import era_cluster_split
from amr_fed.label_fingerprints import CANDIDATES
print('imports OK | long run:', BR.LONG_ROUNDS, 'rounds, local-only', BR.LONG_LOCAL, 'epochs | Part B arms:', list(BR.PART_B_ARMS))

In [ ]:
# 3) Load cohort + check the 5x2 split
import pandas as pd
from amr_fed.data_loader import load_cohort_frame, PK
from amr_fed.split_diagnostics import heterogeneity_vs_null, history_strata
df = load_cohort_frame()
print('rows:', f'{len(df):,}')
SEEDS = (42, 43, 44)
N_ERAS, PER = 5, 2
ERA_SPLIT = lambda d, s: era_cluster_split(d, n_clients=N_ERAS * PER, n_eras=N_ERAS, seed=s)
era_group = lambda raw: int(raw) // PER
a = ERA_SPLIT(df, 42)
assert a.nunique() == 10 and a.index.is_unique and len(a) == df[PK].nunique()
print('hospital sizes:', a.value_counts().sort_index().tolist())
h = heterogeneity_vs_null(df, a, n_null=2, strata=history_strata(df))
print(f"concept shift {h['concept']:.2f} vs random {h['concept_null']:.2f} (ratio {h['concept_ratio']:.2f}; expected ~1.8)")
print('SMOKE PASS')

In [ ]:
# 4) Run the six arms on era 5x2 (resume-safe)
import json, ray
os.environ.pop('AMR_RNG_PROBE', None)
BASE = dict(seeds=SEEDS, df=df, rounds=8, local_epochs=BR.LOCAL_EPOCHS, local_only_epochs=40, hidden=128)
RES_DIR = SRC.parent / 'results'; RES_DIR.mkdir(exist_ok=True)
STEM = 'amr_era5x2'
local = RES_DIR / f'{STEM}.json'
mirror = Path(f'/content/drive/MyDrive/{STEM}.json') if ON_COLAB else None
src = local if local.exists() else (mirror if mirror and mirror.exists() else None)
res = json.load(open(src)) if src else {}
ARMS = dict(BR.PART_B_ARMS)
ARMS['oracle'] = dict(ARMS['oracle'], personalized_group_of=era_group)   # groups of 2, not 5
kw = dict(BASE, partition_fn=ERA_SPLIT, n_clients=10)
paths = [p for p in (local, mirror) if p]
res = BR.run_part(res, ARMS, prefix='era5x2', kw=kw, checkpoint=lambda r: P.save_json(r, *paths), ray_shutdown=ray.shutdown)
P.save_json(res, *paths)
print('arms done:', [a for a in res if not a.startswith('_')])

In [ ]:
# 5) Report: paired comparisons + where each personalized arm put its weight
print(BR.part_b_report(res))
groups = {}
for s in SEEDS:
    raw = ERA_SPLIT(df, s)
    groups[s] = [era_group(u) for u in pd.factorize(raw)[1]]
print('weights (mean over hospitals and seeds):')
print(BR.part_b_weights(res, groups))